# EEGNet Agnostic vs Focused Classification & Comparison

This notebook compares an electrode, time, and frequency-agnostic EEGNet model (raw 60 channels over the entire epoch) with a focused EEGNet model (special spatial-temporal cluster of electrodes and pre-filtered frequency bands, alpha and theta, fused together). We evaluate classification performance, ROC and PR curves, and extract learned spatial convolutional weights as scalp topomaps using MNE.

In [ ]:
import sys
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import classification_report, roc_curve, precision_recall_curve, auc

NOTEBOOK_DIR = Path.cwd()
SRC_DIR = (
    Path("/content/drive/MyDrive/TFG_Nils_2026/03_Codi")
    if "google.colab" in sys.modules
    else NOTEBOOK_DIR.parent.parent / "src"
)

try:
    from google.colab import drive

    IN_COLAB = True
    drive.mount("/content/drive")
except ImportError:
    IN_COLAB = False

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from br_analysis import (
    BinaryTaskConfig,
    RawEpochSpec,
    detect_project_paths,
    load_raw_epoch_dataset,
    get_task_config,
    load_fused_band_dataset,
    FusedBandSpec,
    TARGET_BANDS_CLUSTERS,
    COMMON_BAND_RANGES,
    get_epochs_grid,
)
from br_analysis.torch_training import (
    TorchTrainConfig,
    cross_validate_torch_model,
    plot_fold_metrics,
    plot_torch_history,
    run_torch_epochs_grid,
    run_torch_time_map,
)
from br_analysis.torch_models import EEGNet

warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("WARNING")
RANDOM_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    print(f"Using CUDA GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available; using CPU.")

In [ ]:
TASK_TAG = "c_vs_ip_eegnet"
paths = detect_project_paths("c_vs_i_eegnet", in_colab=IN_COLAB)
SELECTED_SUBJECTS_CSV = paths.analysis_root / "exploration" / "tables" / "subjectes_seleccionats.csv"
MAX_SUBJECTS = 24

task = get_task_config("c_vs_ip", paths.analysis_root, name=TASK_TAG, analysis_window=(0.0, 1.5))

print(f"Project: {paths.project_dir}")
print(f"Data: {paths.data_dir}")
print(f"Results: {paths.analysis_dir}")
print(f"Tables: {paths.tables_dir}")
print(f"Figures: {paths.figures_dir}")

## 1. Agnostic EEGNet Model (Raw Epochs, All 60 Channels)

The Agnostic model is trained on broad bandpass filtered (1-60 Hz) epochs across all 60 electrodes and the entire analysis window (0.0 to 1.5 seconds post-stimulus).

In [ ]:
spec_raw = RawEpochSpec(picks=None, bandpass=(1.0, 60.0), resample_sfreq=128.0)
dataset_agnostic = load_raw_epoch_dataset(paths, task, spec_raw)
X_agnostic = np.asarray(dataset_agnostic.X, dtype=np.float32)
y = np.asarray(dataset_agnostic.y, dtype=np.int64)
subjects = np.asarray(dataset_agnostic.subjects, dtype=object)
SUBJECT_ID = dataset_agnostic.subject_ids[0]  # P002
ch_names_agnostic = list(dataset_agnostic.channel_names)
times = dataset_agnostic.times

print(f"X_agnostic shape: {X_agnostic.shape} (trials, channels, times)")
print(f"y class distribution: {np.bincount(y)}")

### Agnostic EEGNet Architecture & Flow

```mermaid
graph TD
    A[Raw EEG Epochs: 60 channels x 192 times] --> B[Temporal Conv2d: kernel=64, F1=8]
    B --> C[Depthwise Conv2d: Spatial Filters, groups=F1, D=2]
    C --> D[Separable Conv2d: Spatio-temporal mixing, F2=16]
    D --> E[Average Pooling: 1x8]
    E --> F[Fully Connected Layer]
    F --> G[Classification Logits: Congruent vs Incongruent Pure]
```

In [ ]:
n_channels_agnostic = X_agnostic.shape[1]
n_times = X_agnostic.shape[2]


def make_model_agnostic():
    return EEGNet(
        n_channels=n_channels_agnostic,
        n_times=n_times,
        n_classes=2,
        F1=8,
        D=2,
        F2=16,
        kernel_length=64,
        dropout=0.5,
    )


train_cfg = TorchTrainConfig(
    epochs=80,
    batch_size=32,
    lr=1e-3,
    weight_decay=1e-4,
    patience=12,
    n_splits=5,
    val_size=0.2,
    random_seed=RANDOM_SEED,
    device=DEVICE,
    progress=True,
)

print("Training AGNOSTIC EEGNet model on subject P002...")
metrics_agnostic, history_agnostic = cross_validate_torch_model(
    X_agnostic,
    y,
    make_model_agnostic,
    train_cfg,
    subjects=subjects,
    subject_id=SUBJECT_ID,
    save_weights_dir=paths.cache_dir,
    weights_prefix=f"{TASK_TAG}_agnostic",
    plot_roc_pr=False,
)
metrics_agnostic.to_csv(paths.tables_dir / f"cv_metrics_{TASK_TAG}_agnostic.csv", index=False)
print(metrics_agnostic)

## 2. Focused EEGNet Model (22-channel Fused Theta/Alpha Clusters)

The Focused model trains on the theta-band-filtered frontomedial cluster channels and alpha-band-filtered parieto-occipital cluster channels fused along the spatial axis (resulting in 22 channels: 9 theta + 13 alpha).

In [ ]:
import json

# Define path to cluster definitions
cluster_def_path = paths.analysis_root / "clustering" / "cluster_definitions.json"
cluster_defs = None
if cluster_def_path.exists():
    with open(cluster_def_path, "r") as f:
        cluster_defs = json.load(f)
    print(f"Loaded cluster definitions from: {cluster_def_path}")
else:
    print(f"Cluster definitions not found. Using heuristic default TARGET_BANDS_CLUSTERS.")

# Configure target bands and channels
if cluster_defs is not None and task.name in cluster_defs:
    task_def = cluster_defs[task.name]
    # Check which bands have significant clusters
    active_bands = {band: info for band, info in task_def.items() if info["channels"]}
    
    if not active_bands:
        print("WARNING: No significant data-driven clusters found for this task! Falling back to heuristic defaults.")
        # Fallback to defaults
        target_bands = TARGET_BANDS_CLUSTERS
        tmin_crop, tmax_crop = 0.0, 1.5
    else:
        # Use data-driven cluster coordinates
        target_bands = {band: info["channels"] for band, info in active_bands.items()}
        # Take union of time windows
        tmins = [info["tmin"] for info in active_bands.values()]
        tmaxs = [info["tmax"] for info in active_bands.values()]
        tmin_crop = min(tmins)
        tmax_crop = max(tmaxs)
        print(f"Using data-driven bands: {list(active_bands.keys())}")
        print(f"Selected Time Window: {tmin_crop:.3f}s to {tmax_crop:.3f}s")
else:
    # Default fallback
    target_bands = TARGET_BANDS_CLUSTERS
    tmin_crop, tmax_crop = 0.0, 1.5

# Crop task config to the selected time window dynamically
task_cropped = BinaryTaskConfig(
    name=task.name,
    class_0=task.class_0,
    class_1=task.class_1,
    event_codes=task.event_codes,
    selected_subjects_csv=task.selected_subjects_csv,
    max_subjects=task.max_subjects,
    undersample_classes=task.undersample_classes,
    analysis_window=(tmin_crop, tmax_crop),
    random_seed=task.random_seed
)

# Load dataset
spec_focused = FusedBandSpec(
    target_bands_clusters=target_bands, band_ranges=COMMON_BAND_RANGES, channel_mode=("fused",)
)
dataset_focused = load_fused_band_dataset(paths, task_cropped, spec_focused)
X_focused = np.asarray(dataset_focused.X["fused"], dtype=np.float32)
ch_names_focused = dataset_focused.channel_names["fused"]

print(f"X_focused shape: {X_focused.shape} (trials, channels, times) [Channels: {len(ch_names_focused)}]")

### Focused EEGNet Architecture & Flow

```mermaid
graph TD
    A[Raw EEG Epochs] --> B1[Theta bandpass filter: 5-7 Hz]
    A --> B2[Alpha bandpass filter: 8-12 Hz]
    B1 --> C1[Crop Frontomedial Cluster: 9 channels]
    B2 --> C2[Crop Parieto-Occipital Cluster: 13 channels]
    C1 --> D[Concatenate / Fuse: Fused 22 channels]
    C2 --> D
    D --> E[EEGNet Model: n_channels=22]
    E --> F[Classification Output: Congruent vs Incongruent Pure]
```

In [ ]:
n_channels_focused = X_focused.shape[1]
n_times_focused = X_focused.shape[2]


def make_model_focused():
    return EEGNet(
        n_channels=n_channels_focused,
        n_times=n_times_focused,
        n_classes=2,
        F1=8,
        D=2,
        F2=16,
        kernel_length=64,
        dropout=0.5,
    )


## 3. Performance & Spatial Importance Comparison

We compare classification reports, overlaid ROC and PR curves, and plot the learned spatial filters' importances as scalp topomaps for both models.

In [ ]:
# Extract probability predictions
# For simplicity, we recreate folds or compute predictions using cross_validate_torch_model output metrics.
# Let's re-run evaluation or build overlaid ROC/PR plots by loading the saved weights.
from br_analysis.torch_training import build_dataloader, standardize_train_test, _evaluate, temporal_stratified_splits

device = torch.device(DEVICE)
splits = list(temporal_stratified_splits(y[subjects == SUBJECT_ID], train_cfg.n_splits))
X_sub_agn = X_agnostic[subjects == SUBJECT_ID]
X_sub_foc = X_focused[subjects == SUBJECT_ID]
y_sub = y[subjects == SUBJECT_ID]

probs_agn = []
probs_foc = []
y_true_all = []

for fold, (train_idx, test_idx) in enumerate(splits, start=1):
    _, X_test_agn = standardize_train_test(X_sub_agn[train_idx], X_sub_agn[test_idx])
    _, X_test_foc = standardize_train_test(X_sub_foc[train_idx], X_sub_foc[test_idx])
    y_test = y_sub[test_idx]
    y_true_all.extend(y_test)

    # Load Agnostic model weights
    model_agn = make_model_agnostic().to(device)
    model_agn.load_state_dict(torch.load(paths.cache_dir / f"{TASK_TAG}_agnostic_fold{fold}.pt", map_location=device))
    loader_agn = build_dataloader(X_test_agn, y_test, batch_size=train_cfg.batch_size, shuffle=False, device=device)
    _, _, _, p_agn = _evaluate(model_agn, loader_agn, device)
    probs_agn.extend(p_agn)

    # Load Focused model weights
    model_foc = make_model_focused().to(device)
    model_foc.load_state_dict(torch.load(paths.cache_dir / f"{TASK_TAG}_focused_fold{fold}.pt", map_location=device))
    loader_foc = build_dataloader(X_test_foc, y_test, batch_size=train_cfg.batch_size, shuffle=False, device=device)
    _, _, _, p_foc = _evaluate(model_foc, loader_foc, device)
    probs_foc.extend(p_foc)

y_true_all = np.array(y_true_all)
probs_agn = np.array(probs_agn)
probs_foc = np.array(probs_foc)

print("=== Agnostic Model Classification Report ===")
print(classification_report(y_true_all, (probs_agn >= 0.5).astype(int), target_names=[task.class_0, task.class_1]))

print("=== Focused Model Classification Report ===")
print(classification_report(y_true_all, (probs_foc >= 0.5).astype(int), target_names=[task.class_0, task.class_1]))

# Plot overlaid curves
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(12, 5))

# ROC
fpr_agn, tpr_agn, _ = roc_curve(y_true_all, probs_agn)
fpr_foc, tpr_foc, _ = roc_curve(y_true_all, probs_foc)
ax_roc.plot(fpr_agn, tpr_agn, label=f"Agnostic (AUC = {auc(fpr_agn, tpr_agn):.3f})", color="blue")
ax_roc.plot(fpr_foc, tpr_foc, label=f"Focused (AUC = {auc(fpr_foc, tpr_foc):.3f})", color="red")
ax_roc.plot([0, 1], [0, 1], "k--")
ax_roc.set_title("ROC Curve Comparison")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.legend()

# PR
prec_agn, rec_agn, _ = precision_recall_curve(y_true_all, probs_agn)
prec_foc, rec_foc, _ = precision_recall_curve(y_true_all, probs_foc)
ax_pr.plot(rec_agn, prec_agn, label=f"Agnostic (AP = {auc(rec_agn, prec_agn):.3f})", color="blue")
ax_pr.plot(rec_foc, prec_foc, label=f"Focused (AP = {auc(rec_foc, prec_foc):.3f})", color="red")
ax_pr.set_title("Precision-Recall Curve Comparison")
ax_pr.set_xlabel("Recall")
ax_pr.set_ylabel("Precision")
ax_pr.legend()

plt.tight_layout()
roc_pr_path = paths.figures_dir / f"roc_pr_comparison_{TASK_TAG}.png"
plt.savefig(roc_pr_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"ROC and PR curves saved to: {roc_pr_path}")

In [ ]:
def extract_spatial_importance(make_model_fn, weights_prefix):
    importances = []
    for fold in range(1, 6):
        model = make_model_fn()
        model.load_state_dict(torch.load(paths.cache_dir / f"{weights_prefix}_fold{fold}.pt", map_location="cpu"))
        w = model.block1[2].weight.detach().numpy()  # (F1*D, 1, n_channels, 1)
        w_abs = np.abs(np.squeeze(w))  # (F1*D, n_channels)
        importances.append(np.mean(w_abs, axis=0))
    return np.mean(importances, axis=0)


imp_agnostic = extract_spatial_importance(make_model_agnostic, f"{TASK_TAG}_agnostic")
imp_focused = extract_spatial_importance(make_model_focused, f"{TASK_TAG}_focused")

# Plot side-by-side topomaps
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 5))

# Agnostic Info
info_agn = mne.create_info(ch_names=ch_names_agnostic, sfreq=128.0, ch_types="eeg")
montage = mne.channels.make_standard_montage("standard_1005")
info_agn.set_montage(montage, on_missing="ignore")

im1, _ = mne.viz.plot_topomap(imp_agnostic, info_agn, axes=ax1, cmap="viridis", show=False)
ax1.set_title("Agnostic EEGNet spatial weight importance\n(All 60 raw channels)")
fig.colorbar(im1, ax=ax1, shrink=0.6, label="Average weight magnitude")

# Focused Info
info_foc = mne.create_info(ch_names=ch_names_focused, sfreq=128.0, ch_types="eeg")
info_foc.set_montage(montage, on_missing="ignore")

im2, _ = mne.viz.plot_topomap(imp_focused, info_foc, axes=ax2, cmap="viridis", show=False)
ax2.set_title("Focused EEGNet spatial weight importance\n(22-channel Fused Theta/Alpha)")
fig.colorbar(im2, ax=ax2, shrink=0.6, label="Average weight magnitude")

plt.tight_layout()
weights_plot_path = paths.figures_dir / f"eegnet_spatial_importance_{TASK_TAG}.png"
plt.savefig(weights_plot_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Spatial filter topomaps saved to: {weights_plot_path}")

## 4. Agnostic EEGNet Epochs Grid and Sliding Time Window Map

We run CV for different total epoch sizes to evaluate the learning curve, and slide a time window to discover active temporal intervals. 

> [!IMPORTANT]
> **Computational Saving Trick:**
> Instead of executing 100 label-shuffling permutations (which would take several hours to train), we calculate and overlay the **analytical binomial chance level** at a 95% confidence interval based on the test sample size. For a test set of size $N$ under $H_0$, correct classifications follow a binomial distribution $B(N, 0.5)$. The Q95 chance threshold is `stats.binom.ppf(0.95, N, 0.5) / N`.

In [ ]:
epochs_grid = get_epochs_grid("c_vs_ip")
print(f"Running Epochs Grid for {epochs_grid} epochs on subject {SUBJECT_ID}...")
df_grid = run_torch_epochs_grid(
    X_agnostic,
    y,
    make_model_agnostic,
    train_cfg,
    epochs_grid,
    n_repeats=5,  # 5 repeats per grid point
    subjects=subjects,
    subject_id=SUBJECT_ID,
)
df_grid.to_csv(paths.tables_dir / f"epochs_grid_metrics_{TASK_TAG}.csv", index=False)

# Plot learning curve
plt.figure(figsize=(7, 4.5))
plt.plot(df_grid["n_epochs_total"], df_grid["acc_real_mean"], "o-", label="EEGNet Real Accuracy", color="blue")
plt.fill_between(
    df_grid["n_epochs_total"],
    df_grid["acc_real_mean"] - df_grid["acc_real_std"],
    df_grid["acc_real_mean"] + df_grid["acc_real_std"],
    alpha=0.15,
    color="blue",
)
plt.plot(df_grid["n_epochs_total"], df_grid["acc_chance_mean"], "k--", label="Chance Baseline (50%)")
plt.plot(df_grid["n_epochs_total"], df_grid["acc_chance_q95"], "r--", label="Analytical Chance Level (Q95)")
plt.title(f"Agnostic EEGNet Epochs Grid - Subject {SUBJECT_ID}")
plt.xlabel("Total epochs loaded")
plt.ylabel("Classification Accuracy")
plt.legend()
plt.grid(True, linestyle=":")
plt.tight_layout()
epochs_plot_path = paths.figures_dir / f"epochs_grid_{TASK_TAG}.png"
plt.savefig(epochs_plot_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Epochs grid plot saved to: {epochs_plot_path}")

In [ ]:
from br_analysis.experiments import TimeWindowGrid

sfreq = 128.0
win_len = int(0.5 * sfreq)
step_len = int(0.1 * sfreq)
windows_spec = TimeWindowGrid(X_agnostic.shape[2], win_len, step_len, sfreq=sfreq)

print(f"Running Sliding Time-window mapping on subject {SUBJECT_ID}...")
map_results = run_torch_time_map(
    X_agnostic, y, make_model_agnostic, train_cfg, windows_spec, subjects=subjects, subject_id=SUBJECT_ID
)
np.savez_compressed(paths.cache_dir / f"time_map_results_{TASK_TAG}.npz", **map_results)

# Plot time-window accuracy map
plt.figure(figsize=(7, 4.5))
plt.plot(map_results["centers"], map_results["real_map"], "s-", label="EEGNet Real Accuracy", color="purple")
plt.plot(map_results["centers"], map_results["surr_mean"], "k--", label="Chance Baseline (50%)")
plt.plot(map_results["centers"], map_results["surr_q95"], "r--", label="Analytical Chance Level (Q95)")
plt.axvline(0.0, color="black", alpha=0.5, linestyle=":", label="Stimulus Onset")
plt.title(f"Agnostic EEGNet Time-window Accuracy Map - Subject {SUBJECT_ID}")
plt.xlabel("Window center time (s)")
plt.ylabel("Classification Accuracy")
plt.legend()
plt.grid(True, linestyle=":")
plt.tight_layout()
time_plot_path = paths.figures_dir / f"time_map_{TASK_TAG}.png"
plt.savefig(time_plot_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Time-window map saved to: {time_plot_path}")